## Spark Connect on EMR-on-EC2 — Amazon SageMaker Unified Studio Notebook ##
This notebook connects from an Amazon SageMaker Unified Studio notebook to a Spark Connect session running on an Amazon EMR on EC2 cluster, using the project's execution role.

### Security considerations

Amazon SageMaker Unified Studio manages the Spark Connect session for you, so the authentication token and endpoint are not exposed in this notebook. You are still responsible for the surrounding controls (AWS Well-Architected Security Pillar):

- **Authentication mechanism:** the pre-initialized `spark` object is authenticated by SageMaker Unified Studio using the **project/domain execution role** and a short-lived session token that the environment manages and rotates. Verify which execution role backs your project before running workloads.
- **IAM least privilege:** scope that execution role to only the EMR session APIs and the specific S3 prefixes this job needs — no wildcard actions or resources. Restrict the role's trust policy to the intended principals.
- **Data protection at rest :** enable S3 Block Public Access, default server-side encryption (SSE-KMS recommended), and a bucket policy restricting access to the execution role for the target bucket. The demo bucket name below is a placeholder — replace it with your own secured bucket.
- **Detective controls:** enable AWS CloudTrail for EMR and S3 API activity, S3 server access logging / CloudTrail data events on the bucket, and Amazon GuardDuty for anomaly detection.
- **Network protection:** run the EMR cluster in a VPC with security groups scoped to the Spark Connect port, and prefer AWS PrivateLink over public-internet paths.
- **Availability:** the session is managed by the environment; still avoid leaving long-running idle sessions and configure an EMR session idle timeout.

In [0]:
# The `spark` session is created and authenticated by SageMaker Unified Studio using the
# project execution role (short-lived, environment-managed credentials). Confirm which
# execution role backs this project and that it follows least-privilege before running jobs.
spark

Create session for connection: spark-connect-cluster-working.spark


Waiting for EMR on EC2 session to be ready...


Session created for connection: spark-connect-cluster-working.spark.


In [0]:
df = spark.range(10)
df.show()


import os
import pyspark.sql.functions as F

# Target the bucket/prefix via an environment variable so no account-specific path is
# hardcoded. Ensure the bucket has Block Public Access on, SSE-KMS encryption, and a
# bucket policy restricting access to the project execution role (SEC08).
path = os.environ.get("DEMO_S3_PATH", "s3://amzn-s3-demo-smus/smus-demo/")

# 1) Create a small DataFrame on the cluster
df = spark.range(100).withColumn(
    "category", F.when(F.col("id") % 2 == 0, "even").otherwise("odd")
)
print("Before write:")
df.show(5)

# 2) Write to S3 as Parquet (Spark runs on the cluster, writes to S3 directly)
df.write.mode("overwrite").parquet(path)

# 3) Read it back and print
print("After read:")
spark.read.parquet(path).groupBy("category").count().show()


+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



Before write:


+---+--------+
| id|category|
+---+--------+
|  0|    even|
|  1|     odd|
|  2|    even|
|  3|     odd|
|  4|    even|
+---+--------+
only showing top 5 rows


After read:


+--------+-----+
|category|count|
+--------+-----+
|    even|   50|
|     odd|   50|
+--------+-----+

